In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np


base_path = r'D:\BMSCE\Internships\Microsoft Azure AI\IDC_regular_ps50_idx5' 
image_paths = []
labels = []

for patient_folder in os.listdir(base_path):
    patient_path = os.path.join(base_path, patient_folder)
    if os.path.isdir(patient_path):
        for label in ['0', '1']: 
            label_path = os.path.join(patient_path, label)
            if os.path.exists(label_path):
                for img in os.listdir(label_path):
                    if img.endswith('.png'):
                        image_paths.append(os.path.join(label_path, img))
                        labels.append(label)

data = pd.DataFrame({'path': image_paths, 'label': labels})
data = data.sample(n=10000, random_state=42) 

train_data, temp_data = train_test_split(data, test_size=0.3, stratify=data['label'], random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, stratify=temp_data['label'], random_state=42)

datagen = ImageDataGenerator(rescale=1./255)
train_gen = datagen.flow_from_dataframe(train_data, x_col='path', y_col='label', target_size=(50, 50), batch_size=64, class_mode='binary')
val_gen = datagen.flow_from_dataframe(val_data, x_col='path', y_col='label', target_size=(50, 50), batch_size=64, class_mode='binary', shuffle=False)
test_gen = datagen.flow_from_dataframe(test_data, x_col='path', y_col='label', target_size=(50, 50), batch_size=64, class_mode='binary', shuffle=False)

model = Sequential([Conv2D(16, (3, 3), activation='relu', input_shape=(50, 50, 3)), MaxPooling2D((2, 2)), Conv2D(32, (3, 3), activation='relu'), MaxPooling2D((2, 2)), Flatten(),Dense(64, activation='relu'),   Dense(1, activation='sigmoid')])


model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(train_gen, epochs=5, validation_data=val_gen)  # Fewer epochs

test_loss, test_accuracy = model.evaluate(test_gen)
print(f"\nTest Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

test_preds = model.predict(test_gen)
test_preds = (test_preds > 0.5).astype(int)
test_true = test_gen.labels

cm = confusion_matrix(test_true, test_preds)
print("\nConfusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()
print("\nPerformance Analysis:")
print(f"True Negatives (Benign correctly classified): {tn}")
print(f"False Positives (Benign misclassified as Malignant): {fp}")
print(f"False Negatives (Malignant misclassified as Benign): {fn}")
print(f"True Positives (Malignant correctly classified): {tp}")

accuracy = (tn + tp) / (tn + fp + fn + tp)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"\nAccuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1_score:.4f}")


print("\nClassification Report:")
print(classification_report(test_true, test_preds, target_names=['Benign', 'Malignant']))
model.save('models/cancer_classifier_model.h5')


Found 7000 validated image filenames belonging to 2 classes.
Found 1500 validated image filenames belonging to 2 classes.
Found 1500 validated image filenames belonging to 2 classes.


C:\Users\Admin\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\Admin\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 110s 976ms/step - accuracy: 0.7525 - loss: 0.5376 - val_accuracy: 0.7907 - val_loss: 0.4668
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 14s 124ms/step - accuracy: 0.8137 - loss: 0.4264 - val_accuracy: 0.8167 - val_loss: 0.4333
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 14s 123ms/step - accuracy: 0.8253 - loss: 0.4025 - val_accuracy: 0.8247 - val_loss: 0.3951
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 14s 122ms/step - accuracy: 0.8276 - loss: 0.4002 - val_accuracy: 0.8453 - val_loss: 0.3704
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 14s 123ms/step - accuracy: 0.8322 - loss: 0.3869 - val_accuracy: 0.8440 - val_loss: 0.3644
24/24 ━━━━━━━━━━━━━━━━━━━━ 18s 792ms/step - accuracy: 0.8328 - loss: 0.3752

Test Accuracy: 0.8293
Test Loss: 0.3821
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 90ms/step



Confusion Matrix:
[[1006   85]
 [ 171  238]]

Performance Analysis:
True Negatives (Benign correctly classified): 1006
False Positives (Benign misclassified as Malignant): 85
False Negatives (Malignant misclassified as Benign): 171
True Positives (Malignant correctly classified): 238

Accuracy: 0.8293
Precision: 0.7368
Recall: 0.5819
F1-Score: 0.6503

Classification Report:
              precision    recall  f1-score   support

      Benign       0.85      0.92      0.89      1091
   Malignant       0.74      0.58      0.65       409

    accuracy                           0.83      1500
   macro avg       0.80      0.75      0.77      1500
weighted avg       0.82      0.83      0.82      1500

